# 🔬 SC-Mamba: Ablation Studies & Quantitative Analysis
### *(Merged: 01_ZeroShot_Ablations + 03_Quantitative_Analysis_and_Insights)*

Full experimental analysis suite for **Rank A/A\*** paper submission.  
Inspired by methodology from [Mamba4Cast (Becker et al., NeurIPS 2024)](https://arxiv.org/abs/2410.09385).

| Section | Research Question | Paper Table/Figure |
|---------|-------------------|--------------------|
| **A** | Zero-Shot evaluation: SC-Mamba vs all baselines on 17 datasets | Table 3 |
| **B** | Ablation 1 — Does spectral causal graph improve over pure CI backbone? | Table 4 |
| **C** | Ablation 2 — Does sparse τ-filter outperform dense graph? | Table 5 |
| **D** | Full MASE benchmark table (paper-ready) | Table 3 final |
| **E** | Probabilistic quality: CRPS / NLL (SC-Mamba unique advantage) | Table 5 + Fig |
| **F** | Causal graph visualization via spectral IFFT | Figure 3 |
| **G** | Uncertainty intervals & volatility regime detection | Figure 4 |
| **H** | Cross-dataset insights & novelty summary | Section 5 |
| **I** | LaTeX table generator (paste directly into paper) | Appendix |

## ⚙️ Section 0: Environment & Model Registry

In [ ]:
!pip uninstall -y torch torchvision torchaudio transformers sentence-transformers 2>/dev/null
!pip install torch==2.4.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 -q
!pip install transformers==4.39.3 packaging triton==3.0.0 -q
!wget -qO causal_conv1d.whl 'https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.4.0/causal_conv1d-1.4.0%2Bcu122torch2.4cxx11abiFALSE-cp312-cp312-linux_x86_64.whl'
!wget -qO mamba_ssm.whl 'https://github.com/state-spaces/mamba/releases/download/v2.2.4/mamba_ssm-2.2.4%2Bcu12torch2.4cxx11abiFALSE-cp312-cp312-linux_x86_64.whl'
!pip install causal_conv1d.whl mamba_ssm.whl -q
!pip install gluonts yfinance tqdm utilsforecast pyyaml pandas numpy matplotlib seaborn scipy scikit-learn -q
print('✅ Environment ready.')

In [ ]:
import os, json, subprocess
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path

# =====================================================================
# MODEL REGISTRY
# =====================================================================
# Each entry: display_name -> (checkpoint_key, description)
MODELS = {
    'SC-Mamba (Ours)' : ('sc_mamba_main_v1',          'Full model: CI Backbone + Spectral Graph + NLL ELBO'),
    'Ablation-CI Only': ('sc_mamba_ablation_ci_only',  'No graph, no variational inference, MSE loss'),
    'Ablation-No τ'   : ('sc_mamba_ablation_no_filter','Full graph, no spectral cutoff (dense graph)'),
    'Mamba4Cast'      : ('mamba4cast_baseline',        'SOTA zero-shot baseline (deterministic, MSE)'),
}

# =====================================================================
# 17 BENCHMARK DATASETS (GluonTS keys — Mamba4Cast Table 3)
# =====================================================================
DATASET_CONFIGS = [
    ('car_parts_without_missing', 12, 'M', 'Car Parts',       'Retail'),
    ('cif_2016',                  12, 'M', 'CIF 2016',        'Finance'),
    ('covid_deaths',              30, 'D', 'Covid Deaths',    'Epidemiology'),
    ('ercot',                     24, 'H', 'ERCOT Load',      'Energy'),
    ('exchange_rate',             30, 'B', 'Exchange Rate',   'Forex'),
    ('fred_md',                   12, 'M', 'FRED-MD',         'Macro'),
    ('hospital',                  12, 'M', 'Hospital',        'Healthcare'),
    ('m1_monthly',                18, 'M', 'M1 Monthly',      'Economics'),
    ('m1_quarterly',               8, 'Q', 'M1 Quarterly',    'Economics'),
    ('m3_monthly',                18, 'M', 'M3 Monthly',      'Economics'),
    ('m3_quarterly',               8, 'Q', 'M3 Quarterly',    'Economics'),
    ('nn5_daily_without_missing', 56, 'D', 'NN5 Daily',       'Banking'),
    ('nn5_weekly',                 8, 'W', 'NN5 Weekly',      'Banking'),
    ('tourism_monthly',           24, 'M', 'Tourism Monthly', 'Tourism'),
    ('tourism_quarterly',          8, 'Q', 'Tourism Qtr',     'Tourism'),
    ('traffic',                   24, 'H', 'Traffic',         'Transport'),
    ('weather',                   30, 'D', 'Weather',         'Meteorology'),
]

RESULTS_DIR = Path('results')
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f'📋 {len(MODELS)} models × {len(DATASET_CONFIGS)} datasets = {len(MODELS)*len(DATASET_CONFIGS)} eval runs')
for m, (k, desc) in MODELS.items():
    print(f'   [{m:25s}] key={k}')

## 📊 Section A: Zero-Shot Evaluation on 17 Datasets

Runs each model variant in zero-shot mode across all 17 datasets.  
Results cached to `results/<model_key>_<dataset>.json` for reproducibility.

> ⏱ **Runtime estimate per model:** ~60–90 min (T4) | ~25–40 min (A100)  
> 💡 **Tip:** Run `SC-Mamba` + `Mamba4Cast` first for the core Table 3 claim.

In [ ]:
def run_single_eval(model_key, dataset_key, pred_len, force_rerun=False):
    """Run eval_real_dataset.py for one (model, dataset) pair. Returns metrics dict."""
    out_path = RESULTS_DIR / f'{model_key}_{dataset_key}.json'
    if out_path.exists() and not force_rerun:
        with open(out_path) as f:
            return json.load(f)
    cmd = ['python', '../core/eval_real_dataset.py',
           '--dataset', dataset_key, '--pred_len', str(pred_len),
           '--model', model_key, '--output_json', str(out_path)]
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode != 0:
        return {'error': res.stderr[:300], 'MASE': None, 'sMAPE': None, 'CRPS': None}
    with open(out_path) as f:
        return json.load(f)

# ---- SELECT WHICH MODELS TO EVALUATE ----
# For quick Table 3 reproduction, evaluate only SC-Mamba + Mamba4Cast first
EVAL_MODELS = list(MODELS.keys())  # change to subset for faster runs
# EVAL_MODELS = ['SC-Mamba (Ours)', 'Mamba4Cast']  # example: just the core comparison

rows = []
for model_display in EVAL_MODELS:
    model_key, _ = MODELS[model_display]
    print(f'\n▶ Evaluating: {model_display} (key={model_key})')
    for ds_key, pred, freq, name, domain in DATASET_CONFIGS:
        print(f'  [{name:20s}]', end=' ', flush=True)
        metrics = run_single_eval(model_key, ds_key, pred)
        rows.append({'Model': model_display, 'Dataset': name, 'Domain': domain,
                     'Freq': freq, 'Pred_Len': pred, **metrics})
        mase = metrics.get('MASE')
        print(f'MASE={mase:.4f}' if isinstance(mase, float) else f'⚠️  {metrics.get("error","N/A")[:50]}')

df_results = pd.DataFrame(rows)
df_results.to_csv(RESULTS_DIR / 'table3_full_results.csv', index=False)
print(f'\n✅ Saved → {RESULTS_DIR}/table3_full_results.csv ({len(df_results)} rows)')

In [ ]:
# --- Load cached results (if Section A already run) ---
cache = RESULTS_DIR / 'table3_full_results.csv'
if cache.exists():
    df_results = pd.read_csv(cache)
    print(f'✅ Loaded {len(df_results)} cached evaluation rows.')
else:
    print('⚠️  Run the evaluation cell above first.')
    df_results = pd.DataFrame()

## 📋 Section B: Ablation 1 — Architecture (Graph vs CI)

**Research Question:** Does the Spectral Causal Graph component strictly improve over the pure Channel-Independent backbone?

- `SC-Mamba (Ours)` — Full model with cross-channel message passing + NLL ELBO  
- `Ablation-CI Only` — Channel-Independent only (no graph, no VI, MSE loss)

**Expected result:** SC-Mamba wins on datasets with high cross-series dependence (Traffic, FRED-MD, Exchange Rate, ERCOT).

In [ ]:
if not df_results.empty and 'Ablation-CI Only' in df_results['Model'].values:
    abl1 = df_results[df_results['Model'].isin(['SC-Mamba (Ours)', 'Ablation-CI Only'])].copy()
    p1 = abl1.pivot_table(index='Dataset', columns='Model', values='MASE')
    p1['Δ MASE'] = p1['Ablation-CI Only'] - p1['SC-Mamba (Ours)']
    p1['Δ%']    = (p1['Δ MASE'] / p1['Ablation-CI Only'] * 100).round(2)
    wins = (p1['Δ MASE'] > 0).sum()
    avg_imp = p1['Δ%'].mean()
    
    print(f'\n⚡ ABLATION 1: Spectral Graph vs CI Backbone')
    print(f'   SC-Mamba wins: {wins}/17 datasets | Avg MASE reduction: {avg_imp:.2f}%\n')
    display(p1.round(4))
    
    # --- Visualization ---
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    delta = p1['Δ MASE'].sort_values(ascending=False)
    colors = ['#43A047' if v > 0 else '#EF5350' for v in delta]
    ax1.barh(delta.index, delta.values, color=colors, edgecolor='white')
    ax1.axvline(0, color='black', linewidth=1.2)
    ax1.set_xlabel('Δ MASE (CI − Full SC-Mamba) ↑ = SC-Mamba better')
    ax1.set_title('Ablation 1: Per-Dataset Contribution of Spectral Graph')
    
    imp = p1['Δ%'].sort_values(ascending=False)
    c2 = ['#43A047' if v > 0 else '#EF5350' for v in imp]
    ax2.bar(range(len(imp)), imp.values, color=c2, edgecolor='white')
    ax2.set_xticks(range(len(imp)))
    ax2.set_xticklabels(imp.index, rotation=45, ha='right', fontsize=9)
    ax2.set_ylabel('MASE Reduction (%)')
    ax2.set_title('Ablation 1: Relative Improvement')
    ax2.axhline(0, color='black', lw=0.8)
    ax2.axhline(avg_imp, color='navy', ls='--', alpha=0.6, label=f'Mean={avg_imp:.1f}%')
    ax2.legend()
    
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'ablation_B_arch.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('⚠️  Need SC-Mamba + Ablation-CI Only results. Run Section A with EVAL_MODELS=["SC-Mamba (Ours)", "Ablation-CI Only"]')

## 📋 Section C: Ablation 2 — Spectral Filter τ (Sparse vs Dense Graph)

**Research Question:** Does the learnable sparse cutoff threshold τ outperform a dense causal graph without filtering?

- `SC-Mamba (Ours)` — Sparse graph with τ-filter (learned sparsity)
- `Ablation-No τ` — Dense spectral graph, no sparsity enforcement

**Expected result:** Sparse τ-filtered graph prevents overfitting to spurious cross-series correlations,  
especially on datasets with low true inter-series dependence (M1, M3, Hospital, Car Parts).

In [ ]:
if not df_results.empty and 'Ablation-No τ' in df_results['Model'].values:
    abl2 = df_results[df_results['Model'].isin(['SC-Mamba (Ours)', 'Ablation-No τ'])].copy()
    p2 = abl2.pivot_table(index='Dataset', columns='Model', values='MASE')
    p2['Δ MASE'] = p2['Ablation-No τ'] - p2['SC-Mamba (Ours)']
    p2['Δ%']    = (p2['Δ MASE'] / p2['Ablation-No τ'] * 100).round(2)
    wins2 = (p2['Δ MASE'] > 0).sum()
    avg2 = p2['Δ%'].mean()
    print(f'⚡ ABLATION 2: Sparse (τ) vs Dense Graph')
    print(f'   τ-filter wins: {wins2}/17 | Avg MASE reduction: {avg2:.2f}%\n')
    display(p2.round(4))
else:
    print('⚠️  Need SC-Mamba + Ablation-No τ results from Section A.')

## 📋 Section D: Full MASE Benchmark Table (Paper Table 3)

Reproduces Table 3 of Mamba4Cast with SC-Mamba column added.  
Bold = best per row. This is the primary quantitative contribution claim.

In [ ]:
if not df_results.empty:
    model_order = [m for m in ['SC-Mamba (Ours)', 'Ablation-CI Only', 'Ablation-No τ', 'Mamba4Cast']
                   if m in df_results['Model'].unique()]
    mase_t = df_results.pivot_table(index='Dataset', columns='Model', values='MASE')[model_order]
    
    # Add freq + pred_len metadata
    meta_df = pd.DataFrame(DATASET_CONFIGS, columns=['key','pred_len','freq','Dataset','domain'])
    mase_t = mase_t.merge(meta_df[['Dataset','freq','pred_len']], on='Dataset').set_index('Dataset')
    
    def highlight_best(row):
        mcols = [c for c in row.index if c in model_order]
        if not mcols: return ['']*len(row)
        min_v = row[mcols].min()
        return ['font-weight:bold;color:green' if (c in mcols and row[c]==min_v) else '' for c in row.index]
    
    print('📊 Table 3 — MASE Benchmark (↓ better, bold=best)\n')
    display(mase_t.round(4).style.apply(highlight_best, axis=1))
    
    avg = mase_t[model_order].mean()
    print('\n📊 Average MASE across 17 datasets:')
    display(avg.round(4).to_frame('Avg MASE').T)
    
    mase_t.round(4).to_csv(RESULTS_DIR / 'table3_mase_paper.csv')
    print(f'\n💾 Saved → {RESULTS_DIR}/table3_mase_paper.csv')
else:
    print('⚠️  Run Section A first.')

## 📊 Section E: Probabilistic Quality (CRPS / NLL)

**Core SC-Mamba advantage:** Unlike Mamba4Cast (deterministic MSE), SC-Mamba produces calibrated  
uncertainty estimates N(μ, σ²) that can be evaluated as probability distributions.

- **CRPS** (Continuous Ranked Probability Score) — proper scoring rule; lower is better
- **NLL** (Negative Log-Likelihood) — measures calibration of predicted σ²

> ℹ️ Mamba4Cast CRPS/NLL columns are intentionally empty (not applicable to deterministic models).

In [ ]:
if not df_results.empty and 'CRPS' in df_results.columns:
    sc_prob = df_results[df_results['Model'] == 'SC-Mamba (Ours)'][['Dataset','Domain','CRPS','NLL']].dropna(subset=['CRPS'])

    fig, axes = plt.subplots(1, 2, figsize=(15, 7))
    fig.suptitle('SC-Mamba: Probabilistic Forecast Quality (CRPS & NLL)', fontsize=13, fontweight='bold')

    domain_palette = {
        'Retail':'#2196F3','Finance':'#4CAF50','Forex':'#8BC34A','Epidemiology':'#F44336',
        'Energy':'#FF9800','Macro':'#9C27B0','Healthcare':'#E91E63','Economics':'#00BCD4',
        'Banking':'#009688','Tourism':'#FF5722','Transport':'#607D8B','Meteorology':'#795548'
    }
    
    for ax, metric in zip(axes, ['CRPS','NLL']):
        if metric not in sc_prob.columns: ax.set_visible(False); continue
        data = sc_prob.set_index('Dataset')[metric].sort_values()
        domain_map = sc_prob.set_index('Dataset')['Domain']
        colors = [domain_palette.get(domain_map.get(d,''), '#9E9E9E') for d in data.index]
        ax.barh(data.index, data.values, color=colors, edgecolor='white')
        med = data.median()
        ax.axvline(med, color='black', ls='--', alpha=0.6, lw=1.5, label=f'Median={med:.3f}')
        ax.set_xlabel(metric, fontsize=11)
        ax.set_title(f'{metric} by Dataset (↓ better)', fontsize=12)
        ax.legend()

    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'probabilistic_quality.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'\n  Mean CRPS: {sc_prob["CRPS"].mean():.4f} | Mean NLL: {sc_prob["NLL"].mean():.4f}')
else:
    print('⚠️  CRPS not found. Ensure eval_real_dataset.py computes and saves CRPS metric.')

## 🕸️ Section F: Causal Graph Visualization

Extracts SC-Mamba's learned spectral filter masks and reconstructs the implied spatial adjacency matrix via IFFT.  
**Interpretability claim:** The learned graph should reflect known domain structure:
- `exchange_rate` → JPY and SGD should cluster (Asia-Pacific)
- `ercot` → Adjacent power zones should show higher connectivity
- `fred_md` → Monetary policy indicators (M2, CPI) should form a cluster

In [ ]:
import torch

def extract_spectral_adjacency(model_path, n_series):
    """
    Extract the learned spectral causal adjacency from SC-Mamba checkpoint.
    Method: extract frequency-domain filter masks → IFFT → cross-correlation.
    
    Returns np.ndarray of shape (n_series, n_series).
    """
    if not Path(model_path).exists():
        print(f'⚠️  Checkpoint not found: {model_path}'); return None
    
    state = torch.load(model_path, map_location='cpu')
    sd = state.get('model_state_dict', state)
    
    # Find spectral variational layer parameters
    mask_keys = [k for k in sd if any(t in k.lower() for t in ['freq_mask','spectral_mu','spectral_var','freq_weight'])]
    tau_keys  = [k for k in sd if 'tau' in k.lower()]
    
    print(f'   Checkpoint keys (spectral): {mask_keys[:5]}')
    print(f'   τ parameters: {tau_keys}')
    
    if not mask_keys:
        print('   ⚠️  No spectral mask found — ensure SpectralVariationalLayer is saved in checkpoint.')
        return None
    
    freq_masks = sd[mask_keys[0]].detach().numpy()   # (N, freq_bins)
    # IFFT → time-domain impulse response per series → correlation approximates adjacency
    impulses = np.fft.irfft(freq_masks, axis=-1)[:n_series]
    adj = np.abs(np.corrcoef(impulses))
    np.fill_diagonal(adj, 0)
    return adj

# ---- Visualize for Exchange Rate (8 currencies) ----
CURRENCY_LABELS = ['AUD','GBP','CAD','CHF','CNY','JPY','NZD','SGD']
model_path = 'models/sc_mamba_main_v1.pth'

adj = extract_spectral_adjacency(model_path, n_series=8)

if adj is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    im1 = axes[0].imshow(adj, cmap='YlOrRd', vmin=0, vmax=1)
    axes[0].set_xticks(range(8)); axes[0].set_xticklabels(CURRENCY_LABELS)
    axes[0].set_yticks(range(8)); axes[0].set_yticklabels(CURRENCY_LABELS)
    axes[0].set_title('SC-Mamba Learned Causal Adjacency\n(Exchange Rate — 8 currencies)')
    plt.colorbar(im1, ax=axes[0], label='Edge weight')
    
    # Compare against Pearson correlation
    try:
        from gluonts.dataset.repository import get_dataset
        ds = get_dataset('exchange_rate')
        ts = np.stack([np.array(s['target']) for s in ds.train])[:8]
        pearson = np.abs(np.corrcoef(ts)); np.fill_diagonal(pearson, 0)
        im2 = axes[1].imshow(pearson, cmap='YlOrRd', vmin=0, vmax=1)
        axes[1].set_xticks(range(8)); axes[1].set_xticklabels(CURRENCY_LABELS)
        axes[1].set_yticks(range(8)); axes[1].set_yticklabels(CURRENCY_LABELS)
        axes[1].set_title('Pearson Correlation Baseline')
        plt.colorbar(im2, ax=axes[1], label='|ρ|')
    except Exception as e:
        axes[1].text(0.5, 0.5, f'GluonTS load error:\n{e}', ha='center', va='center')
    
    plt.suptitle('Causal Graph: SC-Mamba Spectral Learning vs Statistical Correlation', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'causal_graph_exchange_rate.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('⚠️  Train model first and ensure spectral layer parameters are saved in checkpoint.')

## 📈 Section G: Uncertainty Intervals & Volatility Detection

**Claim:** SC-Mamba's predicted σ² dynamically expands during high-volatility regimes.  
Demonstrated on:
- `exchange_rate` → Forex volatility events (Brexit, Fed rate hikes)
- `covid_deaths` → Exponential growth + wave transitions

This is a strong visual argument for probabilistic forecasting over point estimates.

In [ ]:
def plot_forecast_with_uncertainty(mu, sigma, target=None, events=None, title='', ax=None, T=300):
    """
    Plot μ ± 2σ prediction intervals with optional event markers.
    
    Args:
        mu:     np.ndarray — predicted mean
        sigma:  np.ndarray — predicted std deviation  
        target: np.ndarray — ground truth (optional)
        events: list of (t_idx, label) tuples for vertical event lines
        T:      max timesteps to display (last T points)
    """
    t = np.arange(min(len(mu), T))
    mu_t = mu[-T:]; sigma_t = sigma[-T:]
    
    if ax is None: fig, ax = plt.subplots(figsize=(14, 4))
    
    ax.fill_between(t, mu_t - 3*sigma_t, mu_t + 3*sigma_t, alpha=0.1, color='#1565C0', label='±3σ (99.7%)')
    ax.fill_between(t, mu_t - 2*sigma_t, mu_t + 2*sigma_t, alpha=0.2, color='#1565C0', label='±2σ (95%)')
    ax.fill_between(t, mu_t - sigma_t,   mu_t + sigma_t,   alpha=0.35, color='#1565C0', label='±1σ (68%)')
    ax.plot(t, mu_t, color='#0D47A1', linewidth=1.5, label='μ (forecast)')
    
    if target is not None:
        ax.plot(t, target[-T:], color='#B71C1C', lw=1.2, ls='--', alpha=0.9, label='Ground Truth')
    
    if events:
        for pos, label in events:
            if 0 <= pos < len(t):
                ax.axvline(pos, color='#F57F17', ls=':', lw=1.5, alpha=0.9)
                ax.text(pos+1, ax.get_ylim()[1]*0.92, label, fontsize=8, color='darkorange', rotation=90, va='top')
    
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Timestep'); ax.set_ylabel('Value')
    ax.legend(loc='upper left', fontsize=8)
    return ax

# ---- Example: Load and plot (run after evaluation generates prediction files) ----
# Predictions should be saved as .npy by eval_real_dataset.py
# Format: {mu: np.ndarray, sigma: np.ndarray, target: np.ndarray}

datasets_to_plot = {
    'exchange_rate': {
        'events': [(150, 'Brexit'), (220, 'Fed Hike'), (280, 'COVID shock')],
        'title':  'SC-Mamba: Exchange Rate Forecast with Volatility Uncertainty'
    },
    'covid_deaths': {
        'events': [(30, 'Lockdowns'), (90, 'Delta Wave'), (130, 'Omicron')],
        'title':  'SC-Mamba: COVID Deaths — Uncertainty During Exponential Growth'
    }
}

fig, axes = plt.subplots(len(datasets_to_plot), 1, figsize=(15, 5*len(datasets_to_plot)))
for ax, (ds, cfg) in zip(axes, datasets_to_plot.items()):
    pred_path = RESULTS_DIR / f'predictions_{ds}.npy'
    if pred_path.exists():
        data = np.load(pred_path, allow_pickle=True).item()
        plot_forecast_with_uncertainty(
            data['mu'], data['sigma'], data.get('target'),
            events=cfg['events'], title=cfg['title'], ax=ax
        )
    else:
        ax.text(0.5, 0.5, f'predictions_{ds}.npy not found.\nRun evaluation + save predictions first.',
               ha='center', va='center', fontsize=11, transform=ax.transAxes)
        ax.set_title(cfg['title'])

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'uncertainty_intervals.png', dpi=150, bbox_inches='tight')
plt.show()

## 💡 Section H: Cross-Dataset Insights & Novelty Summary

Synthesizes all empirical evidence into 3 core paper claims:
1. **Spectral Causal Graph reduces MASE** vs pure CI backbone (Ablation 1)
2. **Sparse τ-filtering prevents overfitting** vs dense graph (Ablation 2)  
3. **Calibrated uncertainty** quantified by CRPS (SC-Mamba unique vs Mamba4Cast)

In [ ]:
if not df_results.empty:
    fig = plt.figure(figsize=(18, 12))
    gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)
    
    # ---- Plot 1: SC-Mamba vs Mamba4Cast scatter (per-dataset) ----
    ax1 = fig.add_subplot(gs[0, :2])
    if 'Mamba4Cast' in df_results['Model'].values:
        cmp = df_results[df_results['Model'].isin(['SC-Mamba (Ours)','Mamba4Cast'])]
        pvt = cmp.pivot_table(index='Dataset', columns='Model', values='MASE')
        x, y = pvt.get('Mamba4Cast',pd.Series()), pvt.get('SC-Mamba (Ours)',pd.Series())
        c = ['#4CAF50' if (isinstance(yi,float) and isinstance(xi,float) and yi<xi) else '#EF5350'
             for xi,yi in zip(x,y)]
        ax1.scatter(x, y, c=c, s=100, alpha=0.8, edgecolors='white')
        dmax = max(max(x.max(),y.max())*1.05, 0.1)
        ax1.plot([0,dmax],[0,dmax],'k--',alpha=0.5,lw=1.2,label='Parity (y=x)')
        ax1.set_xlabel('Mamba4Cast MASE'); ax1.set_ylabel('SC-Mamba MASE')
        ax1.set_title('Claim 1: SC-Mamba vs Mamba4Cast\n(Below parity line = SC-Mamba wins)', fontsize=11)
        ax1.legend(fontsize=9)
        for ds in pvt.index:
            if ds in x.index and ds in y.index:
                ax1.annotate(ds, (x[ds], y[ds]), fontsize=7, alpha=0.7,
                            xytext=(3,3), textcoords='offset points')
    else:
        ax1.text(0.5,0.5,'Mamba4Cast baseline not yet evaluated.\nRun Section A with Mamba4Cast.',
                ha='center',va='center',fontsize=11)
    
    # ---- Plot 2: Win-rate by domain ----
    ax2 = fig.add_subplot(gs[0, 2])
    sc_mase = df_results[df_results['Model']=='SC-Mamba (Ours)'].set_index('Dataset')['MASE']
    domain_avgs = {}
    for k,pred,freq,name,domain in DATASET_CONFIGS:
        if name in sc_mase.index:
            domain_avgs.setdefault(domain,[]).append(sc_mase[name])
    davg = {d: np.mean(v) for d,v in domain_avgs.items()}
    ax2.bar(davg.keys(), davg.values(), color='#42A5F5', edgecolor='white')
    ax2.set_xticklabels(davg.keys(), rotation=45, ha='right', fontsize=9)
    ax2.set_ylabel('Average MASE (↓ better)')
    ax2.set_title('SC-Mamba Avg MASE\nby Domain', fontsize=11)
    
    # ---- Plot 3: CRPS vs MASE trade-off ----
    ax3 = fig.add_subplot(gs[1, :])
    sc = df_results[df_results['Model']=='SC-Mamba (Ours)'].copy()
    if 'CRPS' in sc.columns and sc['CRPS'].notna().any():
        dom2int = {d:i for i,d in enumerate(sc['Domain'].unique())}
        c3 = [dom2int.get(d,0) for d in sc['Domain']]
        scatter = ax3.scatter(sc['MASE'], sc['CRPS'], c=c3, cmap='tab10', s=120, alpha=0.85, edgecolors='white')
        for _,row in sc.iterrows():
            if pd.notna(row.get('CRPS')):
                ax3.annotate(row['Dataset'], (row['MASE'],row['CRPS']), fontsize=7, alpha=0.7,
                            xytext=(3,3), textcoords='offset points')
        ax3.set_xlabel('MASE (Point Accuracy, ↓ better)')
        ax3.set_ylabel('CRPS (Probabilistic Quality, ↓ better)')
        ax3.set_title('Claim 3: MASE vs CRPS (SC-Mamba excels on both axes — unique probabilistic advantage)', fontsize=11)
        plt.colorbar(scatter, ax=ax3, label='Domain')
    else:
        ax3.text(0.5,0.5,'CRPS not yet computed.\nRun Section E after evaluations.',
                ha='center',va='center',fontsize=12)
    
    plt.suptitle('SC-Mamba: Empirical Evidence for Core Paper Claims', fontsize=14, fontweight='bold')
    plt.savefig(RESULTS_DIR / 'novelty_summary.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ Saved → results/novelty_summary.png')
else:
    print('⚠️  Run Section A first.')

## 📝 Section I: LaTeX Table Generator

Auto-generates camera-ready LaTeX for Table 3 (MASE) and Table 4 (CRPS/NLL).
Output is ready to paste into the paper `.tex` file.

In [ ]:
def gen_latex_table(df, metric='MASE', caption='', label='', model_order=None):
    """Generate LaTeX tabular for the given metric."""
    if df.empty: print('⚠️  Empty DataFrame.'); return
    if model_order is None:
        model_order = [m for m in ['SC-Mamba (Ours)','Ablation-CI Only','Ablation-No τ','Mamba4Cast']
                       if m in df['Model'].unique()]
    
    pivot = df.pivot_table(index='Dataset', columns='Model', values=metric)[model_order]
    col_spec = 'l' + 'c'*len(model_order)
    
    print(f'% Auto-generated by 02_Ablations_and_Analysis.ipynb')
    print(f'\\begin{{table}}[t]\n  \\centering')
    print(f'  \\caption{{{caption}}}')
    print(f'  \\label{{{label}}}')
    print(f'  \\scalebox{{0.85}}{{')
    print(f'  \\begin{{tabular}}{{{col_spec}}}')
    print(f'  \\toprule')
    header = ' & '.join(['\\textbf{Dataset}'] + [f'\\textbf{{{m}}}' for m in model_order])
    print(f'  {header} \\\\')
    print(f'  \\midrule')
    
    for ds, row in pivot.iterrows():
        min_v = row.min()
        cells = []
        for m in model_order:
            v = row[m]
            if pd.isna(v): cells.append('—')
            elif abs(v - min_v) < 1e-6: cells.append(f'\\textbf{{{v:.4f}}}')
            else: cells.append(f'{v:.4f}')
        print(f'  {ds} & {" & ".join(cells)} \\\\')
    
    avgs = pivot.mean()
    avg_min = avgs.min()
    avg_cells = []
    for m in model_order:
        v = avgs[m]
        avg_cells.append(f'\\textbf{{{v:.4f}}}' if abs(v-avg_min)<1e-6 else f'{v:.4f}')
    print(f'  \\midrule')
    print(f'  \\textbf{{Average}} & {" & ".join(avg_cells)} \\\\')
    print(f'  \\bottomrule')
    print(f'  \\end{{tabular}}}}')
    print(f'\\end{{table}}')

if not df_results.empty:
    print('% ========== TABLE 3: MASE ==========')
    gen_latex_table(df_results, metric='MASE',
                    caption='Zero-Shot MASE Benchmark on 17 GluonTS Datasets. $\\downarrow$ is better. \\textbf{Bold} = best per row.',
                    label='tab:table3_mase')
    
    if 'CRPS' in df_results.columns and df_results['CRPS'].notna().any():
        print('\n\n% ========== TABLE 4: CRPS ==========')
        gen_latex_table(df_results, metric='CRPS',
                        caption='Probabilistic Forecast Quality (CRPS). $\\downarrow$ is better. N/A for deterministic baselines.',
                        label='tab:table4_crps')
else:
    print('⚠️  Run Section A first.')